In [ ]:
import json
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib import rcParams as rc

rc["font.family"] = "Times New Roman"
rc["font.size"] = 14
rc["figure.figsize"] = (6, 4)
rc["axes.grid"] = True

In [ ]:
conf = json.load(open("../data/atem.json"))
times = np.asarray(conf['channels']) * 1e-6
n_turns = conf['n_turns']

In [ ]:
area:str = "NE"
path:str = f"../data/11-024_Alberta_{area}.csv"
dheader:list = [f"zoff30[{i}]" for i in range(30)]
picker:list = ["Line", "bheight", "TranPeak", "x_wgs84", "y_wgs84", "flight", 'pwrline'] + dheader # power line monitor

In [ ]:
dobs = pd.read_csv(path)[picker]

In [ ]:
xy = dobs[["x_wgs84", "y_wgs84"]].to_numpy()
normalizer = (-1e-9)/ (dobs["TranPeak"].values * n_turns).reshape(-1, 1)
dobs[[f"zoff30[{i}]" for i in range(30)]] = dobs[[f"zoff30[{i}]" for i in range(30)]] * normalizer
floors = 5 * normalizer 

In [ ]:
line_no = list(dobs["Line"].unique())

In [ ]:
print(line_no)

In [ ]:
istart:int = 19
index = dobs["Line"] == line_no[istart]
print(f"{line_no[istart]=}")

In [ ]:
plt.scatter(xy[:, 0], xy[:, 1], s=1)
plt.scatter(xy[index, 0], xy[index, 1], s=1)
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title(f"Line {line_no[istart]}")

In [ ]:
dobs.fillna(1e-20)

In [ ]:
plt.figure(figsize=(8, 4),constrained_layout=True)
plt.boxplot(np.abs(dobs[dheader].iloc[index,:])*1e9,\
    showfliers=False,
    showmeans=True
    )
plt.yscale("log")

In [ ]:
dx = 40.
values = []
values_std = []
soundings = []

for i_line, line in enumerate(line_no[istart:istart + 5]):
    df_line = dobs[dobs['Line']==line]

    # Calculate distance along the "Line"
    xy = df_line[["x_wgs84", "y_wgs84"]].to_numpy()
    distance = np.sqrt(((xy-xy[0,:])**2).sum(axis=1))
    max_distance = distance.max()

    # Determine the no. of soundings per bin.
    if max_distance % dx ==0:
        n_sounding = int(max_distance / dx)
    else:
        n_sounding = int(np.round(max_distance / dx) + 1)

    # Create bins and assign each sounding to a bin
    bins = np.arange(n_sounding) * dx
    df_line.insert(0, 'distance', distance)
    # Bin distances
    df_line['bin'] = pd.cut(df_line['distance'], bins=bins)
    # Compute statistics per bin
    binned = (
        df_line.groupby('bin', observed=False)
            [['distance'] + picker[1:]]
            .mean()
    )
    binned.insert(0, 'Line', line)
    binned_std = (
        df_line.groupby('bin', observed=False)
            [['bheight'] + dheader]
            .std()
    )
    values.append(binned.values)
    values_std.append(binned_std.values)
    soundings.append(n_sounding)

df_data_binned = pd.DataFrame(data=np.vstack(values), columns=['Line', 'distance'] + picker[1:])
df_data_std_binned = pd.DataFrame(data=np.vstack(values_std), columns=['bheight'] + dheader)

In [ ]:
print(f"{soundings=}")

In [ ]:
df_data_binned

In [ ]:
df_data_std_binned

In [ ]:
test = df_data_binned[df_data_binned['Line']=='L10170']

In [ ]:
plt.plot(times,np.abs(test[dheader].iloc[0,:]))
plt.xscale("log")
plt.yscale("log")

- standard deviation
$$
\sigma=\frac{1}{N}\sqrt{\sum_{i=1^N}{(x_i-\bar{x})^2}}
$$
- field normalized standard deviation (root-mean-squared relative error)
$$
\frac{\sigma}{d}=\frac{1}{N}\sqrt{\sum_{i=1^N}{\frac{(x_i-\bar{x})^2}{x_i^2}}}\approx
$$

In [ ]:
# Extract timeseries data
data = df_data_binned[dheader].values.astype(float)
# Extract field normalized standard deviation (root mean squared relative error)
data_rerr = (df_data_std_binned[dheader].values / np.abs(df_data_binned[dheader].values)).astype(float)

In [ ]:
channel_id = np.tile(np.arange(data.shape[1]), (data.shape[0], 1))

In [ ]:
cut_off = (data_rerr>0.05) * (channel_id>=0)

In [ ]:
data[cut_off] = np.nan
data_rerr[cut_off] = np.nan

In [ ]:
# Plot histogram of relative errors below the cut-off
hi = plt.hist(data_rerr[~cut_off], bins = np.linspace(0,0.05, 100))

# Todo: 
- Data bining (required review) 
- Figure out the meaning of pwrline (power line management) range.
- Inversion